# Phase 3 — Nigerian Pidgin (pcm) fine-tuning, adapted scope

Gated notebook — run stages **in order**, each one is a checkpoint:

| Stage | What | GPU cost |
|---|---|---|
| 0 | Hard license gates (PRD §8) | none |
| 1 | API + schema verification (run 2026-10-17 — keep for provenance) | none |
| 2 | Download primary dataset: `michaelodafe/pidgin-asr-combined` (CC-BY-4.0) | none |
| 3 | Build speaker-disjoint manifests (HF mode; Rexe-licensed rows hard-excluded) | none |
| 4 | Feasibility micro-run (50 iters, ~40 clips) | minutes |
| 5 | Full run — manually gated on Stage 4 results | hours |

**Verified 2026-10-17** (do not re-derive): official driver is
`OpenBMB/VoxCPM scripts/train_voxcpm_finetune.py --config_path <yaml>`;
manifest schema `{"audio": path, "text": ...}` (+ optional `duration`, `dataset_id`);
LoRA recipe in `conf/voxcpm_v2/voxcpm_finetune_lora.yaml`; base checkpoint Apache-2.0.
**Route v2:** CV 25.0 pcm is Data-Collective-only (404 on HF); primary = pidgin-asr-combined
(CC-BY-4.0) — see `docs/pidgin_dataset_sourcing.md` addendum.
Never improvise around a failure — stop and report it.

In [ ]:
# STAGE 0 - HARD GATES. Proceeds only if the dataset-side flags are true.
# (base_checkpoint_license_ok is already true - Apache-2.0 evidence filed in the config.)
import os, sys, subprocess
os.chdir('/content')
BASE = '/content/naturalvoice-ai'
if not os.path.isdir(BASE):
    subprocess.run(['git', 'clone', 'https://github.com/Etzkennyboi/native-voice.git', BASE], check=True)
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)

import yaml
lic = yaml.safe_load(open('configs/pidgin_baseline.yaml', encoding='utf-8'))['license']
print('license_cleared      :', lic['license_cleared'])
print('consent_review_done  :', lic['consent_review_done'])
print('base_checkpoint_ok   :', lic['base_checkpoint_license_ok'], '(cleared 2026-10-17, Apache-2.0)')
ok = lic['license_cleared'] and lic['consent_review_done']
print()
print('GATE:', 'PASS - go to Stage 2' if ok else
      'BLOCKED - flags flip ONLY after the CC-BY-4.0 attribution + consent review is signed off')
assert ok, 'License gates not cleared - training stays blocked (PRD section 8).'

In [ ]:
# STAGE 1 - verification record (run order 2026-10-17).
# Re-runnable at zero GPU cost; its conclusions were used to wire Stages 3-5.
!pip install -q voxcpm==2.0.3
import voxcpm, pathlib
root = pathlib.Path(voxcpm.__file__).resolve().parent
print('voxcpm root:', root)
!voxcpm --help | head -5
print()
print('verified facts (from package + OpenBMB/VoxCPM GitHub, 2026-10-17):')
print(' - training/ subpackage: data loaders, packers, Accelerator, TrainingState')
print(' - official driver: scripts/train_voxcpm_finetune.py --config_path <yaml>')
print(' - manifest: {"audio": path, "text": str} + optional duration, dataset_id')
print(' - recipe conf/voxcpm_v2/voxcpm_finetune_lora.yaml: LoRA r32 a32 d0.0 (LM+DiT),')
print('   batch 2 x accum 8, lr 1e-4, wd 0.01, warmup 100, 1000 iters, 16k in / 48k out')
print(' - base checkpoint openbmb/VoxCPM2: Apache-2.0 (HF card + repo LICENSE)')

In [ ]:
# STAGE 2 - download the pinned primary dataset (michaelodafe/pidgin-asr-combined, CC-BY-4.0).
# Ungated: no token needed. ~0.95 GB (~9 min of studio-quality read Pidgin news narration).
from huggingface_hub import snapshot_download
local = snapshot_download(
    'michaelodafe/pidgin-asr-combined',
    repo_type='dataset',
    local_dir='/content/pidgin_hf',
)
print('downloaded to:', local)
!find /content/pidgin_hf -maxdepth 3 -type f | sort

In [ ]:
# STAGE 3 - build speaker-disjoint manifests (HF mode). The script re-checks the gates.
# Excludes the license-unspecified 'Rexe' rows by default; unpacks audio to /content/pidgin_hf_clips.
!python scripts/build_pidgin_manifests.py --hf-parquet-dir /content/pidgin_hf --audio-dir /content/pidgin_hf_clips
!wc -l data/manifests/pidgin_train.jsonl data/manifests/pidgin_val.jsonl data/manifests/pidgin_test.jsonl

In [ ]:
# STAGE 4 - feasibility micro-run: OFFICIAL driver, ~40 clips, 50 iterations.
# Purpose: prove the recipe runs here and measure VRAM/time before any paid A100 session.
import os, json, time, pathlib, subprocess, torch, yaml
cfg = yaml.safe_load(open('configs/pidgin_baseline.yaml', encoding='utf-8'))
assert cfg['license']['license_cleared'] and cfg['license']['consent_review_done'], 'clear Stage 0 first'

# official repo = driver scripts + model code
if not os.path.isdir('/content/OpenBMB-VoxCPM'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/OpenBMB/VoxCPM.git', '/content/OpenBMB-VoxCPM'], check=True)

from huggingface_hub import snapshot_download
pretrained = snapshot_download('openbmb/VoxCPM2')
print('pretrained_path:', pretrained)

mini, mini_path = [], 'data/manifests/pidgin_train-mini.jsonl'
for line in open('data/manifests/pidgin_train.jsonl', encoding='utf-8'):
    mini.append(json.loads(line))
    if len(mini) >= 40:
        break
with open(mini_path, 'w', encoding='utf-8') as fh:
    fh.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in mini)

# official lora recipe + feasibility knobs only (num_iters 50, short warmup)
tr = cfg['training']
ft = {
    'pretrained_path': pretrained,
    'train_manifest': mini_path,
    'sample_rate': 16000, 'out_sample_rate': 48000,
    'batch_size': tr['batch_size'], 'grad_accum_steps': tr['grad_accum_steps'],
    'num_workers': 2, 'preprocessing_num_workers': 2,
    'num_iters': 50, 'max_steps': 50, 'warmup_steps': 20,
    'log_interval': 10, 'valid_interval': 10**9, 'save_interval': 50,
    'learning_rate': tr['learning_rate'], 'weight_decay': tr['weight_decay'],
    'max_batch_tokens': tr['max_batch_tokens'], 'max_grad_norm': tr['max_grad_norm'],
    'save_path': '/content/artifacts/pidgin-feasibility',
    'tensorboard': '/content/logs/pidgin-feasibility',
    'lambdas': {'loss/diff': 1.0, 'loss/stop': 1.0},
    'lora': tr['lora'],
    'hf_model_id': 'openbmb/VoxCPM2',
}
yaml.safe_dump(ft, open('configs/pidgin_feasibility.yaml', 'w'), sort_keys=False)

SEED = 3407
torch.manual_seed(SEED)
t0 = time.time()
ret = subprocess.run(['python', '/content/OpenBMB-VoxCPM/scripts/train_voxcpm_finetune.py',
                      '--config_path', 'configs/pidgin_feasibility.yaml']).returncode
record = {
    'stage': 'pidgin_feasibility', 'date': time.strftime('%Y-%m-%d'), 'seed': SEED,
    'num_iters': 50, 'clips': len(mini), 'driver_returncode': ret,
    'train_wall_s': round(time.time() - t0, 1),
    'vram_peak_gib': round(torch.cuda.max_memory_allocated() / 2**30, 2),
    'gpu': torch.cuda.get_device_name(0),
    'text': 'recipe = official conf/voxcpm_v2/voxcpm_finetune_lora.yaml, driver '
            'scripts/train_voxcpm_finetune.py, voxcpm 2.0.3',
}
(pathlib.Path('experiments') / ('pidgin-feasibility-' + time.strftime('%Y%m%d-%H%M') + '.json'))\
 .write_text(json.dumps(record, indent=2), encoding='utf-8')
print('\n=== feasibility record ===\n' + json.dumps(record, indent=2))
print('\nRULE: if the driver errors or VRAM explodes - STOP, no tweaking around; report it.')

In [ ]:
# STAGE 5 - FULL GATED RUN (paid A100 session expected). Two conditions:
#   1. flip RUN_ALLOWED to True after reading the newest feasibility record
#   2. paste the same gate config; the script re-checks nothing here - you are the gate
RUN_ALLOWED = False
assert RUN_ALLOWED, 'Gate closed: review experiments/pidgin-feasibility-*.json first (PRD section 11).'

import os, json, time, pathlib, subprocess, torch, yaml
cfg = yaml.safe_load(open('configs/pidgin_baseline.yaml', encoding='utf-8'))
tr = cfg['training']
from huggingface_hub import snapshot_download
pretrained = snapshot_download('openbmb/VoxCPM2')

ft = {
    'pretrained_path': pretrained,
    'train_manifest': 'data/manifests/pidgin_train.jsonl',
    'val_manifest': 'data/manifests/pidgin_val.jsonl',
    'sample_rate': 16000, 'out_sample_rate': 48000,
    'batch_size': tr['batch_size'], 'grad_accum_steps': tr['grad_accum_steps'],
    'num_workers': 4, 'preprocessing_num_workers': 4,
    'num_iters': tr['num_iters'], 'max_steps': tr['num_iters'],
    'warmup_steps': tr['warmup_steps'],
    'log_interval': 10, 'valid_interval': 250, 'save_interval': 250,
    'learning_rate': tr['learning_rate'], 'weight_decay': tr['weight_decay'],
    'max_batch_tokens': tr['max_batch_tokens'], 'max_grad_norm': tr['max_grad_norm'],
    'save_path': '/content/artifacts/voxcpm2-pidgin-v0',
    'tensorboard': '/content/logs/pidgin-v0',
    'lambdas': {'loss/diff': 1.0, 'loss/stop': 1.0},
    'lora': tr['lora'],
    'hf_model_id': 'openbmb/VoxCPM2',
}
yaml.safe_dump(ft, open('configs/pidgin_voxcpm2_ft_v0.yaml', 'w'), sort_keys=False)

SEED = 3407
torch.manual_seed(SEED)
run_log = {
    'checkpoint_id': 'voxcpm2-pidgin-v0', 'base_revision': cfg['base_model']['revision'],
    'seed': SEED, 'config': 'configs/pidgin_voxcpm2_ft_v0.yaml',
    'train_rows': sum(1 for _ in open('data/manifests/pidgin_train.jsonl', encoding='utf-8')),
    'lora': tr['lora'], 'gpu': torch.cuda.get_device_name(0),
    'started': time.strftime('%Y-%m-%d %H:%M:%S'),
}
print(json.dumps(run_log, indent=2))
ret = subprocess.run(['python', '/content/OpenBMB-VoxCPM/scripts/train_voxcpm_finetune.py',
                      '--config_path', 'configs/pidgin_voxcpm2_ft_v0.yaml']).returncode
run_log.update(finished=time.strftime('%Y-%m-%d %H:%M:%S'), driver_returncode=ret,
               vram_peak_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2))
(pathlib.Path('experiments') / 'pidgin-full-run.json').write_text(json.dumps(run_log, indent=2), encoding='utf-8')
print(json.dumps(run_log, indent=2))
print('\nTuned LoRA weights: /content/artifacts/voxcpm2-pidgin-v0')
print('NOW (non-negotiable): copy to Drive, then disconnect+delete runtime:')
print('  from google.colab import drive; drive.mount(\'/content/drive\'); ')
print('  !mkdir -p /content/drive/MyDrive/naturalvoice-evidence')
print('  !cp -r /content/artifacts /content/experiments /content/drive/MyDrive/naturalvoice-evidence/')

## After the run — non-negotiables

1. Copy `artifacts/` + `configs/pidgin_voxcpm2_ft_v0.yaml` + new `experiments/*.json` to Drive **before** disconnecting.
2. `Runtime > Disconnect and delete` (cost rule, PRD §11).
3. Push configs/experiment records to GitHub — **never** weights or audio (attribution note: CC-BY-4.0 “© Rufai et al. 2025 + Odafe 2026, CC-BY-4.0” is respected; we store no raw audio in the repo).
4. A/B benchmark vs untouched baseline per `benchmarks/pidgin_eval/README.md` —
   ≥3 human Pidgin speakers before any 'sounds Pidgin' claim.
5. Load tuned weights for inference: `VoxCPM.from_pretrained(hf_model_id='openbmb/VoxCPM2', lora_weights_path=<checkpoint dir>)`.